<a href="https://colab.research.google.com/github/DannyJung23/ShoreGate/blob/main/notebooks/Prepare_Tongatapu_Inference_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install earthengine-api geemap

import ee
import geemap
import numpy as np
import pandas as pd
import os
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.2 MB/s eta 0:00:00


In [2]:
ee.Authenticate()
ee.Initialize(project='uoa-gisci')

### Set AOI

In [3]:
aoi = ee.Geometry.Rectangle(
    [
        -175.3661011546799,
        -21.28198284420156,
        -175.02167858181292,
        -21.053702480502235,
    ],
    proj="EPSG:4326",
    geodesic=False,
)

print("AOI bounds:", aoi.bounds().coordinates().getInfo())

aoi_map = geemap.Map(
    basemap="Esri.WorldImagery",
    height="600px",
)
aoi_map.centerObject(aoi, 10)
aoi_map.addLayer(aoi, {"color": "yellow"}, "Tongatapu AOI")
aoi_map


AOI bounds: [[[-175.3661011546799, -21.28198284420156], [-175.02167858181292, -21.28198284420156], [-175.02167858181292, -21.053702480502235], [-175.3661011546799, -21.053702480502235], [-175.3661011546799, -21.28198284420156]]]


Map(center=[-21.16784236325663, -175.19388986824544], controls=(WidgetControl(options=['position', 'transparen…

### Fetch Sentinel-1 pre/post-event imagery

In [4]:
S1_PRE_ID = (
    "COPERNICUS/S1_GRD/"
    "S1A_IW_GRDH_1SDV_20200419T170850_"
    "20200419T170915_032199_03B94C_FE9C"
)

S1_POST_ID = (
    "COPERNICUS/S1_GRD/"
    "S1A_IW_GRDH_1SDV_20220127T170901_"
    "20220127T170926_041649_04F469_BB59"
)

s1_pre_image = ee.Image(S1_PRE_ID).clip(aoi)
s1_post_image = ee.Image(S1_POST_ID).clip(aoi)

# Keep only VV and VH in the trained model's band order.
s1_pre = s1_pre_image.select(["VV", "VH"])
s1_post = s1_post_image.select(["VV", "VH"])

print("S1 pre ID:", s1_pre.get("system:id").getInfo())
print("S1 post ID:", s1_post.get("system:id").getInfo())
print("S1 pre bands:", s1_pre.bandNames().getInfo())
print("S1 post bands:", s1_post.bandNames().getInfo())

S1 pre ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20200419T170850_20200419T170915_032199_03B94C_FE9C
S1 post ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20220127T170901_20220127T170926_041649_04F469_BB59
S1 pre bands: ['VV', 'VH']
S1 post bands: ['VV', 'VH']


### Fetch Sentinel-2 pre-event imagery

In [5]:
S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
S2_CLOUD_PROB_COLLECTION = "COPERNICUS/S2_CLOUD_PROBABILITY"

S2_PRE_START = "2021-10-01"
S2_PRE_END = "2021-12-20"

s2_pre_collection = (
    ee.ImageCollection(S2_COLLECTION)
    .filterBounds(aoi)
    .filterDate(S2_PRE_START, S2_PRE_END)
)

print("S2 pre-event scenes:", s2_pre_collection.size().getInfo())

def mask_s2_scl(image):
    scl = image.select("SCL")

    clear = (
        scl.neq(3)    # cloud shadow
        .And(scl.neq(8))   # medium-probability cloud
        .And(scl.neq(9))   # high-probability cloud
        .And(scl.neq(10))  # cirrus
        .And(scl.neq(11))  # snow/ice
    )

    return image.updateMask(clear)

s2_pre_collection_masked = s2_pre_collection.map(mask_s2_scl)

s2_pre = s2_pre_collection_masked.median().clip(aoi)

print("S2 pre bands:", s2_pre.bandNames().getInfo())

S2 pre-event scenes: 64
S2 pre bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


### Fetch Sentinel-2 post-event imagery

In [6]:
S2_POST_START = "2022-01-17"
S2_POST_END = "2022-01-25"  # Exclusive; includes 24 January
CLOUD_PROB_THRESHOLD = 40

s2_post_sr = (
    ee.ImageCollection(S2_COLLECTION)
    .filterBounds(aoi)
    .filterDate(S2_POST_START, S2_POST_END)
)

s2_post_cloud_prob = (
    ee.ImageCollection(S2_CLOUD_PROB_COLLECTION)
    .filterBounds(aoi)
    .filterDate(S2_POST_START, S2_POST_END)
)

join_condition = ee.Filter.equals(
    leftField="system:index",
    rightField="system:index",
)

joined = ee.ImageCollection(
    ee.Join.saveFirst("cloud_probability").apply(
        primary=s2_post_sr,
        secondary=s2_post_cloud_prob,
        condition=join_condition,
    )
).filter(ee.Filter.notNull(["cloud_probability"]))

print("S2 post SR scenes:", s2_post_sr.size().getInfo())
print("S2 post matched scenes:", joined.size().getInfo())

def mask_s2_cloud_probability(image):
    cloud_probability = ee.Image(
        image.get("cloud_probability")
    ).select("probability")

    scl = image.select("SCL")

    scl_clear = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    cloud_clear = cloud_probability.lt(CLOUD_PROB_THRESHOLD)

    return image.updateMask(scl_clear.And(cloud_clear))

s2_post_collection_masked = joined.map(
    mask_s2_cloud_probability
)

s2_post = s2_post_collection_masked.median().clip(aoi)

print("S2 post bands:", s2_post.bandNames().getInfo())

S2 post SR scenes: 8
S2 post matched scenes: 8
S2 post bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


### ESA WorldCover Built-up Masking

In [7]:
WORLD_COVER_COLLECTION = "ESA/WorldCover/v200"

worldcover = ee.ImageCollection(
    WORLD_COVER_COLLECTION
).first().select("Map").clip(aoi)

built_up = worldcover.eq(50).rename("built_up")

print(
    "WorldCover classes inside AOI:",
    worldcover.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9,
        tileScale=4,
    ).getInfo()
)

worldcover_map = geemap.Map(
    basemap="Esri.WorldImagery",
    height="650px",
)
worldcover_map.centerObject(aoi, 10)

worldcover_map.addLayer(
    built_up.selfMask(),
    {"palette": ["red"]},
    "WorldCover Built-up",
)
worldcover_map.addLayer(
    aoi,
    {"color": "yellow"},
    "Tongatapu AOI",
)
worldcover_map

WorldCover classes inside AOI: {'Map': {'10': 1275924.368627451, '20': 71, '30': 1352958.8352941177, '40': 1, '50': 151457, '60': 5564, '80': 6850171.058823531, '90': 14875, '95': 91877}}


Map(center=[-21.16784236325663, -175.19388986824544], controls=(WidgetControl(options=['position', 'transparen…

### Construct features for the dataset

In [8]:
S2_FEATURE_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]

s2_pre_selected = s2_pre.select(S2_FEATURE_BANDS)
s2_post_selected = s2_post.select(S2_FEATURE_BANDS)

s1_change = s1_post.subtract(s1_pre).rename(
    ["S1_change_VV", "S1_change_VH"]
)

s2_change = s2_post_selected.subtract(
    s2_pre_selected
).rename([
    "S2_change_B2",
    "S2_change_B3",
    "S2_change_B4",
    "S2_change_B8",
    "S2_change_B11",
    "S2_change_B12",
])

feature_names = [
    "S1_pre_VV",
    "S1_pre_VH",
    "S1_post_VV",
    "S1_post_VH",
    "S2_pre_B2",
    "S2_pre_B3",
    "S2_pre_B4",
    "S2_pre_B8",
    "S2_pre_B11",
    "S2_pre_B12",
    "S2_post_B2",
    "S2_post_B3",
    "S2_post_B4",
    "S2_post_B8",
    "S2_post_B11",
    "S2_post_B12",
    "S1_change_VV",
    "S1_change_VH",
    "S2_change_B2",
    "S2_change_B3",
    "S2_change_B4",
    "S2_change_B8",
    "S2_change_B11",
    "S2_change_B12",
]

feature_stack = ee.Image.cat([
    s1_pre,
    s1_post,
    s2_pre_selected,
    s2_post_selected,
    s1_change,
    s2_change,
]).rename(feature_names)

actual_names = feature_stack.bandNames().getInfo()

assert len(actual_names) == 24, f"Expected 24 features, got {len(actual_names)}"
assert actual_names == feature_names, "Feature order mismatch"

print("Feature count:", len(actual_names))
for i, name in enumerate(actual_names):
    print(f"{i:02d}: {name}")

Feature count: 24
00: S1_pre_VV
01: S1_pre_VH
02: S1_post_VV
03: S1_post_VH
04: S2_pre_B2
05: S2_pre_B3
06: S2_pre_B4
07: S2_pre_B8
08: S2_pre_B11
09: S2_pre_B12
10: S2_post_B2
11: S2_post_B3
12: S2_post_B4
13: S2_post_B8
14: S2_post_B11
15: S2_post_B12
16: S1_change_VV
17: S1_change_VH
18: S2_change_B2
19: S2_change_B3
20: S2_change_B4
21: S2_change_B8
22: S2_change_B11
23: S2_change_B12


### Check valid pixels

In [9]:
# A pixel is valid only if every input feature has a valid mask.
feature_valid_mask = (
    feature_stack.mask()
    .reduce(ee.Reducer.min())
    .gt(0)
)

valid_built_up = (
    built_up
    .And(feature_valid_mask)
    .rename("valid_built_up")
)

# Calculate areas at 10 m.
area_image = ee.Image.pixelArea().rename("area_m2")

area_stats = (
    area_image
    .updateMask(valid_built_up)
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9,
        tileScale=4,
    )
    .getInfo()
)

valid_area_m2 = area_stats.get("area_m2", 0) or 0

built_area_stats = (
    ee.Image.pixelArea()
    .updateMask(built_up)
    .rename("area_m2")
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9,
        tileScale=4,
    )
    .getInfo()
)

built_area_m2 = built_area_stats.get("area_m2", 0) or 0

print(f"Total Built-up area: {built_area_m2 / 1e6:.3f} km²")
print(f"Valid Built-up area: {valid_area_m2 / 1e6:.3f} km²")
print(
    "Valid percentage of Built-up area:",
    f"{100 * valid_area_m2 / built_area_m2:.2f}%"
    if built_area_m2 else "No Built-up pixels",
)

valid_map = geemap.Map(
    basemap="Esri.WorldImagery",
    height="650px",
)
valid_map.centerObject(aoi, 10)

valid_map.addLayer(
    built_up.selfMask(),
    {"palette": ["red"]},
    "WorldCover Built-up",
)
valid_map.addLayer(
    valid_built_up.selfMask(),
    {"palette": ["cyan"]},
    "Valid pixels for RF",
)
valid_map.addLayer(aoi, {"color": "yellow"}, "AOI")

valid_map

Total Built-up area: 14.043 km²
Valid Built-up area: 8.613 km²
Valid percentage of Built-up area: 61.34%


Map(center=[-21.167842363256664, -175.193889868246], controls=(WidgetControl(options=['position', 'transparent…

In [10]:
preview_map = geemap.Map(
    basemap="Esri.WorldImagery",
    height="650px",
)
preview_map.centerObject(aoi, 10)

preview_map.addLayer(
    s2_pre,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000, "gamma": 1.2},
    "S2 pre median",
)
preview_map.addLayer(
    s2_post,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000, "gamma": 1.2},
    "S2 post median",
)
preview_map.addLayer(
    valid_built_up.selfMask(),
    {"palette": ["cyan"]},
    "Valid RF pixels",
)
preview_map.addLayer(aoi, {"color": "yellow"}, "AOI")

preview_map


Map(center=[-21.167842363256664, -175.193889868246], controls=(WidgetControl(options=['position', 'transparent…

### Download the dataset

In [13]:
!pip -q install rasterio

import os
import ee
import geemap
import rasterio
from rasterio.merge import merge
from google.colab import files

EXPORT_NAME = "tongatapu_rf_24features_builtup.tif"
LOCAL_OUTPUT_PATH = f"/content/{EXPORT_NAME}"
TILE_DIR = "/content/tongatapu_rf_tiles"
os.makedirs(TILE_DIR, exist_ok=True)

EXPORT_CRS = "EPSG:32701"
EXPORT_SCALE = 10
NODATA_VALUE = -9999
TILE_SIZE_M = 5000  # Smaller tiles to reduce request size

# Reuse the feature stack and valid pixel mask created earlier.
rf_export_image = (
    feature_stack
    .toFloat()
    .updateMask(valid_built_up)
    .unmask(NODATA_VALUE)
    .clip(aoi)
)

# Transform AOI with a non-zero error margin before getting bounds.
projected_aoi = aoi.transform(
    ee.Projection(EXPORT_CRS),
    maxError=10
)

projected_bounds = projected_aoi.bounds(
    maxError=10,
    proj=ee.Projection(EXPORT_CRS)
)

coords = projected_bounds.coordinates().getInfo()[0]
xs = [p[0] for p in coords]
ys = [p[1] for p in coords]

xmin, xmax = min(xs), max(xs)
ymin, ymax = min(ys), max(ys)

print("Projected bounds:", xmin, ymin, xmax, ymax)

tiles = []
x = xmin

while x < xmax:
    y = ymin
    while y < ymax:
        x2 = min(x + TILE_SIZE_M, xmax)
        y2 = min(y + TILE_SIZE_M, ymax)

        tile_rect = ee.Geometry.Rectangle(
            [x, y, x2, y2],
            proj=EXPORT_CRS,
            geodesic=False
        )

        # Non-zero error margin for geometry intersection.
        tile_region = tile_rect.intersection(
            projected_aoi,
            maxError=10
        )

        tiles.append((tile_region, x, y))
        y += TILE_SIZE_M
    x += TILE_SIZE_M

print("Tiles to export:", len(tiles))

tile_paths = []

for i, (tile_region, x, y) in enumerate(tiles, start=1):
    tile_path = os.path.join(TILE_DIR, f"tile_{i:03d}.tif")

    print(f"Downloading tile {i}/{len(tiles)}")

    geemap.ee_export_image(
        rf_export_image,
        filename=tile_path,
        scale=EXPORT_SCALE,
        crs=EXPORT_CRS,
        region=tile_region,
        file_per_band=False
    )

    if not os.path.exists(tile_path) or os.path.getsize(tile_path) == 0:
        raise FileNotFoundError(
            f"Tile {i} was not downloaded successfully: {tile_path}"
        )

    tile_paths.append(tile_path)

print("Successfully downloaded tiles:", len(tile_paths))

Projected bounds: 669505.226868 7645357.571049 705561.475484 7671039.197188
Tiles to export: 48
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_001.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_002.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_003.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_004.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_005.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_006.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_007.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_008.tif
Generating URL ...
Please wait ...
Data downloaded to /content/tongatapu_rf_tiles/tile_009.tif
Generating URL ...
Please wait ...
Data downloade

In [14]:
if not tile_paths:
    raise RuntimeError("No downloaded tiles found. Run Cell 11 first.")

src_files = [rasterio.open(path) for path in tile_paths]

try:
    mosaic, mosaic_transform = merge(
        src_files,
        nodata=NODATA_VALUE
    )

    profile = src_files[0].profile.copy()
    profile.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "count": mosaic.shape[0],
        "dtype": "float32",
        "crs": EXPORT_CRS,
        "transform": mosaic_transform,
        "nodata": NODATA_VALUE,
        "compress": "deflate",
        "tiled": True,
    })

    with rasterio.open(LOCAL_OUTPUT_PATH, "w", **profile) as dst:
        dst.write(mosaic.astype("float32"))

finally:
    for src in src_files:
        src.close()

with rasterio.open(LOCAL_OUTPUT_PATH) as src:
    print("Output:", LOCAL_OUTPUT_PATH)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("NoData:", src.nodata)
    print(f"File size: {os.path.getsize(LOCAL_OUTPUT_PATH) / 1024**2:.2f} MB")

    assert src.count == 24, f"Expected 24 bands, got {src.count}"

files.download(LOCAL_OUTPUT_PATH)


Output: /content/tongatapu_rf_24features_builtup.tif
Bands: 24
CRS: EPSG:32701
Resolution: (10.0, 10.0)
NoData: -9999.0
File size: 6.67 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>